In [20]:
### 🧮 Python Script


import pandas as pd
import numpy as np
import random
from faker import Faker
import os

os.getcwd()
os.chdir(r'C:\GYANENDRA\INFORMATION_TECHNILOGY_PROJECTS\CREDIT_RISK_ENGINE')

fake = Faker()



In [15]:

# Base parameters
N_CLIENTS = 10000
AVG_LOANS_PER_CLIENT = 2
OUTPUT_DIR = "CREDIT_RISK_CLIENT_DATA"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------
# 1. Counterparty Profile
# -------------------------------
clients = []
for i in range(N_CLIENTS):
    cid = f"C{str(i+1).zfill(5)}"
    clients.append({
        "ClientID": cid,
        "ClientName": fake.name(),
        "ClientType": random.choice(["Individual", "Corporate"]),
        "Country": random.choice(["India", "UK", "USA", "Singapore", "UAE"]),
        "Sector": random.choice(["IT", "Manufacturing", "Real Estate", "Retail", "Finance"]),
        "NetWorth_Cr": round(random.uniform(5, 500), 2),
        "CreditScore": random.randint(550, 850),
        "RelationshipManager": f"RM{random.randint(100,199)}",
        "ClientAge": random.randint(25,70),
        "AccountTenure_Years": random.randint(1,20)
    })

df_clients = pd.DataFrame(clients)
df_clients.to_csv(os.path.join(OUTPUT_DIR,"counterparty.csv"), sep=";", index=False)


In [16]:
# -------------------------------
# 2. Credit Data (Loans)
# -------------------------------
loans = []
loan_id = 1
for client in df_clients["ClientID"]:
    n_loans = np.random.poisson(AVG_LOANS_PER_CLIENT)
    for _ in range(max(1,n_loans)):
        loans.append({
            "LoanID": f"L{str(loan_id).zfill(6)}",
            "ClientID": client,
            "LoanAmount_Cr": round(random.uniform(1,100),2),
            "InterestRate": round(random.uniform(6,12),2),
            "Tenure_Months": random.choice([12,24,36,48,60]),
            "StartDate": fake.date_between(start_date="-5y", end_date="today"),
            "EndDate": fake.date_between(start_date="today", end_date="+5y"),
            "LoanType": random.choice(["Secured","Unsecured"]),
            "RiskCategory": random.choice(["Low","Medium","High"]),
            "PurposeOfLoan": random.choice(["Business","Personal","Mortgage"])
        })
        loan_id += 1

df_loans = pd.DataFrame(loans)
df_loans.to_csv(os.path.join(OUTPUT_DIR,"credit_data.csv"), sep=";", index=False)


In [17]:
# -------------------------------
# 3. Collateral Details
# -------------------------------
collaterals = []
coll_id = 1
for loan in df_loans["LoanID"]:
    n_coll = random.randint(1,3)
    for _ in range(n_coll):
        val = round(random.uniform(1,200),2)
        exposure = df_loans.loc[df_loans["LoanID"]==loan,"LoanAmount_Cr"].values[0]
        collaterals.append({
            "CollateralID": f"COL{str(coll_id).zfill(6)}",
            "LoanID": loan,
            "CollateralType": random.choice(["Property","Shares","Bonds","Cash"]),
            "CollateralValue_Cr": val,
            "ValuationDate": fake.date_between(start_date="-1y", end_date="today"),
            "HaircutPercent": random.choice([10,20,30]),
            "CollateralCoverageRatio": round(val/exposure,2),
            "CollateralLocation": random.choice(["Mumbai","London","New York","Dubai","Singapore"])
        })
        coll_id += 1

df_coll = pd.DataFrame(collaterals)
df_coll.to_csv(os.path.join(OUTPUT_DIR,"collateral.csv"), sep=";", index=False)


In [18]:

# -------------------------------
# 4. Repayment History
# -------------------------------
repayments = []
rep_id = 1
for loan in df_loans["LoanID"]:
    n_rep = random.randint(3,10)
    for _ in range(n_rep):
        status = random.choice(["Paid","Overdue","Default"])
        repayments.append({
            "RepaymentID": f"R{str(rep_id).zfill(6)}",
            "LoanID": loan,
            "LastPaymentDate": fake.date_between(start_date="-1y", end_date="today"),
            "NextDueDate": fake.date_between(start_date="today", end_date="+1y"),
            "EMI_Amount": round(random.uniform(0.01,0.5),2),
            "OutstandingBalance_Cr": round(random.uniform(0,100),2),
            "PaymentStatus": status,
            "DaysPastDue": random.randint(0,90) if status!="Paid" else 0,
            "RepaymentConsistency": random.choice(["High","Medium","Low"])
        })
        rep_id += 1

df_rep = pd.DataFrame(repayments)
df_rep.to_csv(os.path.join(OUTPUT_DIR,"repayment.csv"), sep=";", index=False)


In [21]:

# -------------------------------
# 5. Trades
# -------------------------------
trades = []
trade_id = 1
for client in df_clients["ClientID"]:
    n_trades = random.randint(1,5)
    for _ in range(n_trades):
        trades.append({
            "TradeID": f"T{str(trade_id).zfill(6)}",
            "ClientID": client,
            "TradeVolume_Cr": round(random.uniform(0,50),2),
            "TradeFrequency": random.randint(1,50),
            "LastTradeDate": fake.date_between(start_date="-1y", end_date="today"),
            "AssetClass": random.choice(["Equity","Bonds","Derivatives"]),
            "MarketExposure_Cr": round(random.uniform(0,100),2),
            "RiskAppetiteScore": random.randint(30,90)
        })
        trade_id += 1

df_trades = pd.DataFrame(trades)
df_trades.to_csv(os.path.join(OUTPUT_DIR,"trades.csv"), sep=";", index=False)


In [19]:

# -------------------------------
# 6. Market Data
# -------------------------------
market = []
for i in range(365):  # one year of daily data
    market.append({
        "Date": fake.date_between(start_date="-1y", end_date="today"),
        "FXExposure_Cr": round(random.uniform(50,200),2),
        "EquityExposure_Cr": round(random.uniform(100,300),2),
        "BondExposure_Cr": round(random.uniform(80,250),2),
        "StressTestScore": random.randint(50,100),
        "VaR_Percent": round(random.uniform(1,10),2),
        "MacroEconomicIndex": f"GDP:{round(random.uniform(5,8),1)},Inflation:{round(random.uniform(3,6),1)},Rate:{round(random.uniform(5,8),1)}"
    })

df_market = pd.DataFrame(market)
df_market.to_csv(os.path.join(OUTPUT_DIR,"market_data.csv"), sep=";", index=False)